# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JasperOwen/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis.** Each row in the raw dataset represents one page's performance on one day, that belongs to one client. For the model input, I will format these daily records into summary records where each row represent's the performance across March 2026 of one page that belongs to one client, with columns that calculate perforamcne change such as drops in clicks


**Time window.** I will use March 2026 as my development month and decision point. This means that I will only use data from that month to identify signs of declining performance and create my refresh priority score. I will use later months only to test if my model's rankings are correct. The data from after March 2026 will not be used when ranking the pages.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

import duckdb
from huggingface_hub import login

login(token=hf_token)

con = duckdb.connect()
con.sql("SET enable_http_metadata_cache=true;")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=hf_token
)

file_path

'/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet'

In [3]:
con.sql(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{file_path}')
    LIMIT 1
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
| ------------------------------------------------------ | -------- | ------------------------------------ |
| `sessions_ai`                                           | Feature  | Shows how many users were directed to a page by AI, which indicates its visibility and how often AI cites it |
| `gsc_clicks`                                      | Feature  | Shows how many users clicked on the page in a Google Search, making it a strong indicator of the page's performance. |
| `gsc_avg_position`                                     | Feature  | Shows how highly the page ranks in Google Search, which influences how likely users are to find it. |
| `ga4_engaged_sessions`                                         | Feature  | Shows how many users actively engaged with the page, indicating the quality of its content. |
| `ga4_total_engagement_sec`                                        | Feature  | Shows how long a user engaged with a page, which indicates how useful the content is |
| `refresh_priority_score`                          | Label    | Used to compare the performances of each page over time, with the worst performing pages having the highest score |
| `client_hash_id`                                       | Context  | Used to identify clients, not to be used when making predictions |
| `content_hash_id`                                      | Context  | Used to identify pages, not to be used when making predictions |
| `report_date`                                          | Context  | Used for aggregating data into monthly summaries only |
| `month`                                                | Context  | Allows me to organise the data by monthly performance and compare page performance across months |
| `ga4_data_available`                                   | Context  | Only indicates if a particular page has ga4 data avaliable, not relevent when making predictions |
| `client_has_gsc`                                       | Context  | Only indicates if a client has gsc data avaliable, not relevent when making predictions |
| `client_has_ga4`                                       | Context  | Only indicates if a client has ga4 data avaliable, not relevent when making predictions |
| Performance metrics past March 2026            | Excluded | These values are being used for testing, letting the model see them beforehand would be data leakage |
| `ga4_*` metrics where `ga4_data_available is not true` | Excluded | The data is unavaliable so it cannot be used |
| Rows where `gsc_data_available is not true`            | Excluded | The data is unavaliable so it cannot be used |

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{file_path}')
    LIMIT 1
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Each of three queries below verify the claims I have made above:

The first query is the grrain check. This confirms that my unit of analysis claim is true: that each row in the dataset is one page's performance in one day, where the page belongs to one client. By grouping the data by client_hash_id, content_hash_id, and report_date and finding no duplicates, I have shown that the grain holds

The second query confirms that the time window claim from seciton 1 is correct: that the development slice is March 2026. This is shown because the min and max dates outputted are March 1st 2026 and March 31st 2026

The third query is the avaliability check which confirms the avaliability of the data I will use with both gsc and ga4 data avaliable. This shows how much data my exclusion rules remove and how much data my model will be able to use.



In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM read_parquet('{file_path}')
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count


In [6]:
con.sql(f"""
SELECT
    COUNT(*)         AS slice_row_count,
    MIN(report_date) AS slice_min_date,
    MAX(report_date) AS slice_max_date
FROM read_parquet('{file_path}')
""").df()

,slice_row_count,slice_min_date,slice_max_date
0,9841378,2026-03-01,2026-03-31


In [7]:
con.sql(f"""
SELECT
    COUNT(*)                                                    AS total_rows,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
    SUM(CASE WHEN (ga4_data_available IS TRUE AND gsc_data_available IS TRUE) THEN 1 ELSE 0 END) AS ga4_and_gsc_available_rows
FROM read_parquet('{file_path}')
""").df()

,total_rows,gsc_available_rows,ga4_available_rows,ga4_and_gsc_available_rows
0,9841378,3611061.0,413966.0,364347.0


In [8]:
feature_frame = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_clicks ELSE 0 END) AS early_march_clicks,
    SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_clicks ELSE 0 END) AS late_march_clicks,

    AVG(CASE WHEN report_date <= '2026-03-15' THEN gsc_avg_position ELSE NULL END) AS early_march_avg_position,
    AVG(CASE WHEN report_date > '2026-03-15' THEN gsc_avg_position ELSE NULL END) AS late_march_avg_position,

    SUM(CASE WHEN report_date <= '2026-03-15' THEN ga4_total_engagement_sec ELSE 0 END) AS early_march_engagement_sec,
    SUM(CASE WHEN report_date > '2026-03-15' THEN ga4_total_engagement_sec ELSE 0 END) AS late_march_engagement_sec

FROM read_parquet('{file_path}')
WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
LIMIT 5
""").df()

feature_frame

,client_hash_id,content_hash_id,early_march_clicks,late_march_clicks,early_march_avg_position,late_march_avg_position,early_march_engagement_sec,late_march_engagement_sec
0,client_9958f0a7ae1df715,content_810cf06597918291,0.0,1.0,8.795464,12.924784,938.0,1233.0
1,client_9958f0a7ae1df715,content_b813c73d7000b3b1,1.0,0.0,7.290612,9.712825,1.0,100.0
2,client_9958f0a7ae1df715,content_5a77dbf5671c5a65,105.0,94.0,4.722870,4.353800,588.0,501.0
3,client_9958f0a7ae1df715,content_278030b007943b07,5.0,2.0,6.463128,5.365841,89.0,35.0
4,client_9958f0a7ae1df715,content_237e63fc00c8c7ad,15.0,24.0,6.955875,7.219061,6.0,11.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The dataset I will be using drops 96.3% of total dataset for March 2026 (364,347 rows used out of 9,841,378) due to using both gsc and ga4 data. The data can never show historical page performance for clients that do not use both gsc and ga4..

In addition, because the training data only uses March 2026, it is blind to long-term patterns. This means it will never be able to know if a decline in performance is true decline, seasonal decline or other pre-determined reasons that the page does not need refreshing for.

Also, the dataset can never note any changes made to the webpages during March 2026, it will only measure changes in features from the first and second halves of the month.

Finally, the data contains no marker for exactly which pages are in decline. Instead the label is a proxy score, meaning the data can never explicitly confirm that a page's performance is in decline.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.